# 04 · Mocking & Isolation

**Focus:** isolating network / disk I/O with `pytest-mock`.

A good unit test is **fast, deterministic, and offline**. But real code calls APIs, reads files,
hits databases. If a test actually makes an HTTP request it becomes slow and flaky — it fails when
the network is down or the remote data changes.

The fix is **mocking**: replace the I/O call with a stand-in that returns a canned result. The
`pytest-mock` plugin gives every test a `mocker` fixture — a thin, auto-cleaned wrapper around
`unittest.mock`. We'll use `mocker.patch(...)` to swap out the HTTP call so **no real request is ever made.**

### Setup — point the shell at this course's tools
The `!` cells below run command-line tools (`pytest`, later `mutmut`, `playwright`). We prepend the
active kernel's `bin/` directory to `PATH` so those commands resolve to the versions installed for
**this course**, not some other Python on your machine. Run this cell first.

In [1]:
import sys, os
# The kernel's own interpreter lives in the course virtualenv's bin/ directory.
os.environ["PATH"] = os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"]
print("Using Python:", sys.executable)
!pytest --version

Using Python: /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson/.venv/bin/python


pytest 9.1.1


### Code that calls an external API
It fetches a user over HTTP with `httpx` and reshapes the response.

In [2]:
%%writefile nb04_api.py
import httpx

API = "https://api.example.com"

def get_username(user_id: int) -> str:
    '''Fetch a user from the API and return an "@handle" string.'''
    resp = httpx.get(f"{API}/users/{user_id}")
    resp.raise_for_status()
    data = resp.json()
    return "@" + data["username"]

Writing nb04_api.py


### The test — patch the network away
We patch `httpx.get` *as seen inside `nb04_api`* and hand back a fake response object whose
`.json()` returns whatever we want. The real `api.example.com` is never contacted.

In [3]:
%%writefile test_nb04.py
from nb04_api import get_username

def test_get_username_uses_api_response(mocker):
    # Build a fake response object.
    fake_resp = mocker.Mock()
    fake_resp.json.return_value = {"username": "ada_lovelace", "id": 42}
    fake_resp.raise_for_status.return_value = None

    # Replace httpx.get inside nb04_api with a mock that returns our fake response.
    mock_get = mocker.patch("nb04_api.httpx.get", return_value=fake_resp)

    result = get_username(42)

    assert result == "@ada_lovelace"                          # used our fake data
    mock_get.assert_called_once_with("https://api.example.com/users/42")  # called the right URL

def test_no_real_network(mocker):
    # If the patch weren't in place, this would try to hit the internet.
    mock_get = mocker.patch("nb04_api.httpx.get")
    mock_get.return_value.json.return_value = {"username": "grace"}
    mock_get.return_value.raise_for_status.return_value = None

    assert get_username(7) == "@grace"
    assert mock_get.called   # proof the mock, not the network, handled the call

Writing test_nb04.py


### Run it — instant, offline, deterministic

In [4]:
!pytest -v test_nb04.py

============================= test session starts ==============================


platform darwin -- Python 3.12.11, pytest-9.1.1, pluggy-1.6.0 -- /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson
plugins: syrupy-5.5.3, mock-3.15.1, cov-7.1.0, xdist-3.8.0, asyncio-1.4.0, time-machine-3.2.0, anyio-4.14.2, base-url-2.1.0, playwright-0.8.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collecting ... 

collected 2 items                                                              

test_nb04.py::test_get_username_uses_api_response PASSED                 [ 50%]
test_nb04.py::test_no_real_network PASSED                                [100%]

============================== 2 passed in 0.08s ===============================


### ⚠️ Common pitfall — patch where it's *used*, not where it's *defined*
`mocker.patch("nb04_api.httpx.get")` works because `nb04_api` looks the name up as `httpx.get`.
Patching `"httpx.get"` directly is fragile, and if the code had done `from httpx import get`, you'd
have to patch `"nb04_api.get"` instead. **Rule of thumb: patch the name in the module that *calls* it.**

### 🔬 Try it yourself
A mock records everything it was called with. Run this to inspect `call_args` and `call_count`, then
**try** changing the expected URL in `assert_called_once_with(...)` to a wrong value and re-run to see
the mock-verification failure.

In [5]:
%%writefile test_nb04_tryit.py
from nb04_api import get_username

def test_inspect_the_mock(mocker):
    fake = mocker.Mock()
    fake.json.return_value = {"username": "linus"}
    fake.raise_for_status.return_value = None
    mock_get = mocker.patch("nb04_api.httpx.get", return_value=fake)

    get_username(99)

    print("call_args :", mock_get.call_args)     # what URL did our code request?
    print("call_count:", mock_get.call_count)
    mock_get.assert_called_once_with("https://api.example.com/users/99")  # <-- change the URL to see it fail

Writing test_nb04_tryit.py


In [6]:
!pytest -v -s test_nb04_tryit.py

============================= test session starts ==============================
platform darwin -- Python 3.12.11, pytest-9.1.1, pluggy-1.6.0 -- /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson
plugins: syrupy-5.5.3, mock-3.15.1, cov-7.1.0, xdist-3.8.0, asyncio-1.4.0, time-machine-3.2.0, anyio-4.14.2, base-url-2.1.0, playwright-0.8.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collecting ... 

collected 1 item                                                               

test_nb04_tryit.py::test_inspect_the_mock call_args : call('https://api.example.com/users/99')
call_count: 1
PASSED

============================== 1 passed in 0.12s ===============================


### What you learned
- `mocker.patch("module.where_its_used.thing")` — patch where the name is *looked up*, not where it's defined.
- A `mocker.Mock()` fakes the response; set `.json.return_value` to control the payload.
- `mock.assert_called_once_with(...)` verifies your code called the dependency correctly.
- The tests run with **no network**, so they're fast and never flaky.

Next up: **determinism and time** — freezing the clock so time-dependent code is testable.